In [1]:
import torch
torch.__version__

'2.8.0+cu129'

# Tensors

## Basics


In [5]:
# create tensor 
scaler = torch.tensor(5) 
vector = torch.tensor([1,2,3])  #.reshape(-1,1)
matrix = torch.tensor([[1,2,3],
                        [4,5,6]], dtype=torch.float)

In [3]:
# print shape and data types 
print(f"Scaler => shape: {scaler.shape},\n dtype: {scaler.dtype} \n")
print(f"Vector => shape: {vector.shape},\n dtype: {vector.dtype} \n")
print(f"Matrix => shape: {matrix.shape},\n dtype: {matrix.dtype}")

Scaler => shape: torch.Size([]),
 dtype: torch.int64 

Vector => shape: torch.Size([3]),
 dtype: torch.int64 

Matrix => shape: torch.Size([2, 3]),
 dtype: torch.float32


In [6]:
# move device of a tensor
print(f" Device of Matrix: {matrix.device} \n")
matrix = matrix.to('cuda:0')
print(f"After change, the device is {matrix.device}")

 Device of Matrix: cpu 

After change, the device is cuda:0


# Autograd

## require_grad

In [7]:
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)
y = w*x 

y.backward() 
w.grad

tensor(3.)

In [8]:
print(x.grad)

None


- Why does `x.grad` not exist?
- Why does `w.grad` exist? 

## Manual grad vs auto grad

In [9]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True) 
x = torch.tensor(3.0) 

y = w*x + b 
y.backward()

- What gradient is $ \frac{\partial y}{\partial w}$? 
<br>Manual 
- What gradient is $ \frac{\partial y}{\partial b}$?

In [10]:
# Manual Value 
w_grad = 1*x + 0.0 
b_grad = 0.0 + 1.0 
print(f"Manual Gradient => Grad W: {w_grad} \n Grad b: {b_grad}")

Manual Gradient => Grad W: 3.0 
 Grad b: 1.0


In [11]:
# Auto grad 
print(f"Autograd => Grad W: {w.grad} \n Grad b: {b.grad}")

Autograd => Grad W: 3.0 
 Grad b: 1.0


In [12]:
y.backward()

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

The error occurs because after calling the `y.backward()` pytorch freed the gradient graph untill we specify to retain the graph. 

## Gradient Accumulation

In [28]:
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0) 

y1 = w*x
y2 = w*x 

y1.backward() 
print(f'After first backward => {w.grad.item()}')

y2.backward() 
print(f"After 2nd backward => {w.grad.item()}")

After first backward => 3.0
After 2nd backward => 6.0


- Why does gradient doubled?
- Why is `zero_grad` needed?

Gradient accumulates in pytorch because `.grad` buffers add new gradients by default.

Enabling - 
- mini-batch training,
- multi-loss optimization, and 
- flexible backpropagation.

# QnA

*Questions and Answers regrading this notebook

[Topic](#require_grad)
- Why does `x.grad` not exist?
- Why does `w.grad` exist? 

[Topic](#gradient-accumulation)
- Why does gradient doubled?
- Why is `zero_grad` needed?

Deep Thinking - 
- Why does pytorch clear the gradient graph after we call `y.backward()` ?
- What are the benefits of gradient accumulation?

## Interactions

### WHY GRADIENT ACCUMULATION EXISTS (THE REAL REASON)
Core idea (this is the key sentence):

- In PyTorch, gradients are added to .grad by default, not overwritten.

- This design is intentional, not accidental.

### MECHANISM (WHAT ACTUALLY HAPPENS INTERNALLY)

Consider this variable:
```python
w = torch.tensor(2.0, requires_grad=True)
```
PyTorch creates:
```bash
w.data → the value

w.grad → a buffer to store gradients
```

***
Now each time you call:
```python
loss.backward()
```
PyTorch does:
```bash
w.grad = w.grad + ∂loss/∂w
```
❗ It does NOT do:
```bash
w.grad = ∂loss/∂w
```
That’s the entire reason gradients accumulate.

### WHY THIS DESIGN IS NECESSARY (NOT OPTIONAL)

#### Reason 1 — Mini-batch training

  - In real training:
```
loss = loss_batch1 + loss_batch2 + loss_batch3
loss.backward()
```
OR equivalently:
```
loss1.backward()
loss2.backward()
loss3.backward()
```
  - Accumulation allows:

Multiple losses

Multiple backward passes

Same final gradient
***
#### Reason 2 — Gradient accumulation to simulate large batch size

  - Used when GPU memory is small:
```python
optimizer.zero_grad()

for i in range(k):
    loss = model(batch_i)
    loss.backward()   # accumulate gradients

optimizer.step()
```
This simulates:
```bash
One big batch = sum of smaller batches
```
This is standard industry practice.
***
#### Reason 3 — Multiple loss functions

 - Example:
```python
loss = loss_cls + 0.1 * loss_reg
loss.backward()
```
Or:
```python
loss_cls.backward(retain_graph=True)
loss_reg.backward()
```
 - Accumulation allows:

Composite objectives

Multi-task learning

CONSEQUENCE (WHAT GOES WRONG IF YOU FORGET)
If you forget zero_grad():
| Iteration | Gradient in `.grad` |
| --------- | ------------------- |
| 1         | g₁                  |
| 2         | g₁ + g₂             |
| 3         | g₁ + g₂ + g₃ ❌      |


➡️ Updates explode
➡️ Loss becomes unstable
➡️ Model “mysteriously” diverges

This is one of the most common beginner bugs.
***
> ONE-SENTENCE INTERVIEW ANSWER (MEMORIZE)

Gradients accumulate in PyTorch because .grad buffers add new gradients by default, <br>enabling mini-batch training, multi-loss optimization, and flexible backpropagation.

## Roughs